# 02 — Phase A feature extraction (teacher-forced PersonaPlex bnb4)

For each dataset clip, replays the conversation through the **live-identical**
bnb4 step path: user codes → `input_tokens`, dataset agent codes → `moshi_tokens`,
word-alignment text stream → `text_token`. Captures per 80ms step:

- `hidden [T, 4096]` fp16 — final-layer `transformer_out` (the same tensor the live server captures)
- `agent_codes [T, 8]` int16 and `mimi_lat [T, 512]` fp16 (stateless RRVQ decode)
- `pros [T, 6]` fp16 DSP prosody, `agent_mask`, `label`, `text_stream`

Decisions:
- `USE_LIVE_PROMPT_PREFIX = True` (default): the Robert text prompt + VARM3 voice
  prefix are applied ONCE and snapshotted, then restored per clip — this matches
  the live hidden-state distribution. VARM3 is NOT used to generate speech here.
- Resume-safe: delete nothing; re-running skips completed clips.
- Time: measured after 5 clips; ~0.8 s/clip on a 5090 → ~1.8 h for 8k clips.


In [ ]:
from pathlib import Path
import os, sys, json, time
import numpy as np
import torch

sys.path.insert(0, os.getcwd())
import eh_common as eh

P = eh.get_paths()   # override: P = eh.get_paths("/your/root") or set EMOTION_HEAD_ROOT
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16
print("ROOT:", P.ROOT, "| device:", DEVICE)
for k in ("MOSHI_BNB4", "MIMI_FILE", "SPM_FILE", "VOICE_PT"):
    p = Path(getattr(P, k))
    print(f"  {k}: {'OK' if p.exists() else 'MISSING'}  {p}")


In [ ]:
MAX_CLIPS = 8000
USE_LIVE_PROMPT_PREFIX = True
CLIPS_PER_SHARD = 256


In [ ]:
import pandas as pd
df = pd.read_parquet(P.DATA / "manifest.parquet")
df = df[df.has_wav].copy()
quota = max(1, MAX_CLIPS // len(eh.EMOTIONS))
# NB: no groupby().apply() here — pandas >= 2.2/3.x drops the grouping column
# from the applied frame (include_groups=False), which silently breaks sel.emotion
_parts = [g.sample(min(len(g), quota), random_state=eh.SEED)
          for _, g in df[df.split == "train"].groupby("emotion")]
assert _parts, "no train rows with has_wav=True — check 01's wav coverage / split columns"
sel = pd.concat(_parts).reset_index(drop=True)
sel["alignments"] = sel["alignments"].map(json.loads)
sel["agent_turn_frames"] = sel["agent_turn_frames"].map(json.loads)
print(sel.emotion.value_counts(), "| total:", len(sel))

AGENT_NAMES = {"agent", "assistant", "moshi", "bot", "personaplex",
               "speaker_broker", "speaker_0", "spk0", "left", "0"}
def agent_side(w):
    spk = str(w.get("speaker", "")).strip().lower()
    return (not spk) or (spk in AGENT_NAMES)


In [ ]:
sp = eh.load_sp(P)
mimi = eh.load_mimi(P, DEVICE)
lm = eh.load_lm_bnb4(P, DEVICE, DTYPE)
print(f"cuda mem after load: {torch.cuda.memory_allocated()/1e9:.2f} GiB")
lm_gen = eh.make_lm_gen(lm, DEVICE)
eh.install_hidden_step(lm_gen)

if USE_LIVE_PROMPT_PREFIX:
    assert P.TEXT_PROMPT_FILE.exists(), \
        f"upload {P.TEXT_PROMPT_FILE} (or set USE_LIVE_PROMPT_PREFIX=False)"
    assert P.VOICE_PT.exists(), f"missing {P.VOICE_PT}"
    prompt_state = eh.apply_prompts_and_snapshot(
        lm_gen, mimi, sp, str(P.VOICE_PT),
        P.TEXT_PROMPT_FILE.read_text(encoding="utf-8", errors="replace"))
    print("LIVE prompt prefix (Robert + VARM3) applied once; state tensors:", len(prompt_state))
else:
    prompt_state = eh.snapshot_streaming_state(lm_gen)
    print("NO prompt prefix (flag off); state tensors:", len(prompt_state))


In [ ]:
# text-stream convention sanity (see 01's interleaver printout for the visual cross-check)
assert eh.TEXT_PAD == 3 and eh.TEXT_EOP == 0
bad = 0
for _, row in sel.head(50).iterrows():
    stream = eh.build_agent_text_stream([w for w in row["alignments"] if agent_side(w)], 400, sp)
    bad += int((stream >= 32000).sum()) + int((stream < 0).sum())
assert bad == 0, "text piece ids out of range"
print("text-stream sanity OK (PAD=3, EOP=0, piece ids < 32000)")


In [ ]:
# PARITY CHECK: same clip twice with the prompt state restored -> hidden must be
# bit-near-identical (cosine > 0.999). Proves the snapshot/restore + teacher-forced
# step path is deterministic and matches the live capture contract.
row = sel.iloc[0]
agent = eh.read_channel(row.wav_path, 0)
user = eh.read_channel(row.wav_path, 1)
n_frames = max(len(agent), len(user)) // eh.FRAME_SAMPLES + 1
stream = eh.build_agent_text_stream([w for w in row["alignments"] if agent_side(w)], n_frames, sp)
hiddens = []
for rep in range(2):
    eh.restore_streaming_state(lm_gen, mimi, prompt_state)
    hiddens.append(eh.extract_clip_features(lm_gen, mimi, agent, user, stream, DEVICE)["hidden"].float())
cos = float(torch.nn.functional.cosine_similarity(hiddens[0].flatten(), hiddens[1].flatten(), dim=0))
print(f"hidden determinism cosine (state restored): {cos:.5f}")
assert cos > 0.999, "hidden path not deterministic across state restore — do NOT proceed"
print("PARITY OK")


In [ ]:
from tqdm.auto import tqdm
log = eh.ProgressLogger(P.FEAT_A / "extract_phaseA.log", echo_every=25)
writer = eh.ShardWriter(P.FEAT_A, "phaseA", CLIPS_PER_SHARD)
todo = [r for r in sel.to_dict("records") if r["clip_id"] not in writer.completed]
total_target = len(writer.completed) + len(todo)
log.log(f"=== phaseA extraction START | completed={len(writer.completed)} "
        f"todo={len(todo)} target={total_target} | device={DEVICE} "
        f"live_prompt={USE_LIVE_PROMPT_PREFIX}", force=True)
t0 = time.time()
failed, times = 0, []
for row in tqdm(todo, total=len(todo)):
    ct0 = time.time()
    try:
        agent = eh.read_channel(row["wav_path"], 0)
        user = eh.read_channel(row["wav_path"], 1)
        n_frames = max(len(agent), len(user)) // eh.FRAME_SAMPLES + 1
        stream = eh.build_agent_text_stream([w for w in row["alignments"] if agent_side(w)], n_frames, sp)
        eh.restore_streaming_state(lm_gen, mimi, prompt_state)
        feats = eh.extract_clip_features(lm_gen, mimi, agent, user, stream, DEVICE)
        T = feats["hidden"].shape[0]
        mask = np.zeros(T, dtype=bool)
        for s, e in row["agent_turn_frames"]:
            mask[max(0, s):min(T, e)] = True
        ue = row.get("user_emotion")
        writer.add({
            "clip_id": row["clip_id"], **feats,
            "agent_mask": torch.from_numpy(mask),
            "label": eh.EMO2ID[row["emotion"]],
            "user_emotion": eh.EMO2ID.get(str(ue).lower(), -1) if ue else -1,
            "user_id": row["user_id"], "split": row["split"],
            "text_stream": torch.from_numpy(stream).to(torch.int32),
        })
    except Exception as e:
        failed += 1
        log.log(f"ERROR clip={row['clip_id']} ({row['emotion']}): "
                f"{type(e).__name__}: {e} | skipped (resume-safe)", force=True)
        continue
    dt = time.time() - ct0
    times.append(dt)
    completed = len(writer.completed)
    remaining = max(0, total_target - completed)
    avg = sum(times) / len(times)
    log.log(f"clip {completed}/{total_target} | +{dt:.2f}s avg={avg:.2f}s "
            f"| left={remaining} ETA={avg * remaining / 3600:.2f}h "
            f"| failed={failed} | {row['clip_id']} ({row['emotion']})")
writer.close()
log.log(f"=== phaseA extraction END | completed={len(writer.completed)}/{total_target} "
        f"failed={failed} | elapsed={(time.time() - t0) / 3600:.2f}h", force=True)
print("done. total completed:", len(writer.completed), "| failed:", failed,
      "| log:", log.path)


In [ ]:
import collections
import matplotlib.pyplot as plt
recs = list(eh.iter_shard_records(P.FEAT_A, "phaseA"))
print("clips on disk:", len(recs))
r = recs[0]
for k, v in r.items():
    if isinstance(v, torch.Tensor):
        print(f"  {k}: {tuple(v.shape)} {v.dtype}")
    else:
        print(f"  {k}: {v}")
print("labels:", collections.Counter(r["label"] for r in recs))
r = recs[min(3, len(recs) - 1)]
fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
axes[0].plot(r["hidden"].float().norm(dim=1))
axes[0].set_title(f"hidden L2 norm — {r['clip_id']} ({eh.EMOTIONS[r['label']]})")
axes[1].plot(r["agent_mask"].float())
axes[1].set_title("agent-speaking mask")
axes[1].set_ylim(-0.1, 1.1)
P.REPORTS.mkdir(parents=True, exist_ok=True)
plt.tight_layout()
plt.savefig(P.REPORTS / "phaseA_feature_sanity.png", dpi=130)
plt.show()
del lm, lm_gen, mimi
eh.free_gpu()
print("→ continue with 03_train_and_evaluate_head_phaseA.ipynb")
